# RealDrift on Sepsis, BPIC2012 and Emergency: four tiers, drift detection and evaluation
Generates the sudden, gradual, hard and recurrent streams of every log, runs PELT and Version Clustering on each stream and scores them against the ground truth.
Run the cells top to bottom. Generated streams are reused when you run again, so an interrupted run continues where it stopped.
Emergency is private. If its files are not on this machine the notebook stops in section 1 (set `REQUIRE_ALL = False` to continue with the other logs).

In [ ]:
import os, sys, subprocess, json, glob
import numpy as np
import pandas as pd
import yaml
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)

CONFIGS = {"sepsis": "configs/sepsis_config.yaml",
           "bpic12": "configs/bpic2012_config.yaml",
           "emergency": "configs/emergency_config.yaml"}
NAMES   = {"sepsis": "Sepsis", "bpic12": "BPIC2012", "emergency": "Emergency"}
TIERS   = ["sudden", "gradual", "hard", "recurrent"]
SEEDS   = [0, 1, 2]            # evaluation seeds, use [0, 1, 2, 3, 4] for five
TUNE    = False                # True tunes detector parameters on separate streams, see section 4
TUNE_SEEDS = [100]
REQUIRE_ALL = True             # stop when a log is not ready instead of silently leaving it out
FORCE   = False                # True regenerates streams that already exist
QUICK   = False                # True is a smoke test with a few cases per instance, not for results
QUICK_MAX_CASES = 300
STREAMS = "outputs/streams"    # evaluation streams go to STREAMS/eval, tuning streams to STREAMS/tune
OUT     = "outputs"            # detection_raw.csv and detection_summary.csv

def run(cmd, tail=3000):
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    out = "\n".join(l for l in (p.stdout + p.stderr).splitlines()
                    if "it/s" not in l and "kde" not in l and "warn" not in l.lower())
    print(out[-tail:])
    if p.returncode != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")
quick = ["--max-cases-per-instance", str(QUICK_MAX_CASES)] if QUICK else []

## 1. Check that every log is ready
A log is ready when its generated pools, its cluster sublogs and either the cached calendars or the raw log are present. Missing parts can be built with `scripts/prepare_dataset.py` (sublogs, calendars) and `src/step4_trace_pool_generation.py` (pools).

In [ ]:
rows = []
for ds, path in CONFIGS.items():
    cfg = yaml.safe_load(open(path))
    K = cfg["n_clusters"]
    pools = [os.path.exists(os.path.join(cfg["concept_pools_dir"], f"generated_{ds}_rank{r}{l}.csv"))
             for r in range(1, K + 1) for l in "AB"]
    subs = [os.path.exists(os.path.join(cfg["sublogs_dir"], f"{ds}_sublog_C{k}.csv")) for k in range(1, K + 1)]
    cal = os.path.exists(cfg["calendars_cache"]) or os.path.exists(cfg["raw_log"])
    rows.append({"dataset": ds, "K": K, "d_supp": cfg["calendar_params"]["d_supp"],
                 "pools": f"{sum(pools)}/{len(pools)}", "sublogs": f"{sum(subs)}/{len(subs)}",
                 "calendars or raw log": cal, "ready": all(pools) and all(subs) and cal})
ready_tbl = pd.DataFrame(rows).set_index("dataset"); display(ready_tbl)
DATASETS = [d for d in CONFIGS if ready_tbl.loc[d, "ready"]]
if REQUIRE_ALL and len(DATASETS) < len(CONFIGS):
    missing = {d: [k for k, v in {"pools": ready_tbl.loc[d, "pools"].split("/")[0] == ready_tbl.loc[d, "pools"].split("/")[1],
                                  "sublogs": ready_tbl.loc[d, "sublogs"].split("/")[0] == ready_tbl.loc[d, "sublogs"].split("/")[1],
                                  "calendars cache or raw log": ready_tbl.loc[d, "calendars or raw log"]}.items() if not v]
               for d in CONFIGS if d not in DATASETS}
    raise RuntimeError(f"not ready, missing parts per log {missing}. Fix this or set REQUIRE_ALL = False.")
assert DATASETS, "no log is ready"
print("running", DATASETS)

exp = yaml.safe_load(open("configs/experiment.yaml"))
exp.update({"configs": [CONFIGS[d] for d in DATASETS], "tiers": TIERS, "seeds": SEEDS,
            "tuning_seeds": TUNE_SEEDS, "out_dir": STREAMS})
os.makedirs(OUT, exist_ok=True)
EXP = f"{OUT}/experiment_notebook.yaml"
yaml.safe_dump(exp, open(EXP, "w"))

## 2. Generate the streams
Every log, tier and seed gives one stream. A full-scale BPIC2012 stream takes about 3 minutes, Sepsis and Emergency are much smaller.

In [ ]:
run([sys.executable, "scripts/run_composition.py", "--experiment", EXP, "--phase", "eval", "--datasets", *DATASETS]
    + (["--force"] if FORCE else []) + quick)

In [ ]:
mf = sorted(glob.glob(f"{STREAMS}/eval/*_manifest.json"))
manifests = [json.load(open(f)) for f in mf]
expected = {(d, t, s) for d in DATASETS for t in TIERS for s in SEEDS}
found = {(m["dataset"], m["tier"], m["seed"]) for m in manifests}
assert expected <= found, f"missing streams {sorted(expected - found)}"
extra = found - expected
assert not extra, (f"{STREAMS}/eval holds streams that are not part of this run {sorted(extra)[:5]} ... "
                   "the detection step would score them too, delete them or change STREAMS")
comp = pd.DataFrame([{"dataset": m["dataset"], "tier": m["tier"], "seed": m["seed"], "n_drifts": len(m["drift_types"]),
                      "placed": m["n_placed"], "excluded": m["n_excluded"],
                      "excluded_share": round(m["n_excluded"] / max(1, m["n_placed"] + m["n_excluded"]), 3),
                      "mean_delay_h": round(m["mean_delay_per_case_h"], 1),
                      "atkde_fallbacks": len(m["arrival_fallback_instances"]),
                      "valid": m["validation"]["ok"] and m["ground_truth_check"]["ok"]} for m in manifests])
assert comp["valid"].all(), "a composed stream failed validation, do not evaluate detectors on it"
display(comp.groupby(["dataset", "tier"]).agg(seeds=("seed", "count"), drifts=("n_drifts", "first"),
        placed=("placed", "mean"), excluded_share=("excluded_share", "mean"),
        mean_delay_h=("mean_delay_h", "mean"), atkde_fallbacks=("atkde_fallbacks", "sum")).round(3))

### Do the streams suit the detector windows?
PELT uses windows of 7 days and VC cuts a log into `n_windows` windows of equal trace count. A stream of only a few weeks (Emergency has cycle times of hours) or very few traces per window gives the detectors little to work with. Warnings point to settings to revisit in `configs/detectors.yaml`.

In [ ]:
det = yaml.safe_load(open("configs/detectors.yaml"))["detectors"]
def settings(name, ds):
    p = dict(det[name]["params"]); p.update((det[name].get("per_dataset") or {}).get(ds) or {}); return p
chk = []
for ds in DATASETS:
    for tier in TIERS:
        pre = f"{STREAMS}/eval/{ds}_{tier}_seed{SEEDS[0]}"
        log = pd.read_csv(pre + "_drift_log.csv")
        tcol = [c for c in log.columns if "time" in c.lower()][0]
        t = pd.to_datetime(log[tcol], format="mixed", utc=True)
        span = (t.max() - t.min()).days
        ncase = log[[c for c in log.columns if "case" in c.lower()][0]].nunique()
        pelt_w = span / settings("pelt", ds)["window_days"]
        vc_n = settings("vc", ds)["n_windows"]
        chk.append({"dataset": ds, "tier": tier, "span_days": span, "cases": ncase,
                    "pelt_windows": round(pelt_w), "vc_traces_per_window": round(ncase / vc_n, 1),
                    "warning": ("few PELT windows " if pelt_w < 20 else "") + ("few traces per VC window" if ncase / vc_n < 10 else "")})
display(pd.DataFrame(chk))

### Ground truth of one stream per tier (first seed)

In [ ]:
import matplotlib.pyplot as plt
for ds in DATASETS:
    fig, axes = plt.subplots(len(TIERS), 1, figsize=(10, 2.2 * len(TIERS)))
    for ax, tier in zip(axes, TIERS):
        pre = f"{STREAMS}/eval/{ds}_{tier}_seed{SEEDS[0]}"
        log = pd.read_csv(pre + "_drift_log.csv")
        tcol = [c for c in log.columns if "time" in c.lower()][0]
        t = pd.to_datetime(log[tcol], format="mixed", utc=True).dt.tz_localize(None)
        rule = "W" if (t.max() - t.min()).days > 150 else "D"
        cnt = pd.Series(1, index=t).resample(rule).sum()
        ax.plot(cnt.index, cnt.values, color="0.4", lw=1)
        for _, r in pd.read_csv(pre + "_transitions.csv").iterrows():
            s, e = [pd.Timestamp(r[k]).tz_convert(None) for k in ("start_ts", "end_ts")]
            ax.axvspan(s, max(e, s + pd.Timedelta(hours=6)), color="red", alpha=.35)
        ax.set_title(f"{NAMES[ds]}, {tier} (seed {SEEDS[0]}), events per {'week' if rule == 'W' else 'day'}, red is ground truth", fontsize=9)
    plt.tight_layout(); plt.show()

## 3. Detectors
PELT and Version Clustering are configured in `configs/detectors.yaml`. The `params` blocks are used as they are, with the `per_dataset` entries on top, unless tuning is switched on below.

In [ ]:
for k, v in det.items():
    for ds in DATASETS:
        print(k, ds, settings(k, ds))
print("tolerances:", yaml.safe_load(open("configs/detectors.yaml"))["tolerances"])

## 4. Optional tuning
Tuning picks, per detector and log, the grid point with the best mean F1 on separate streams (`TUNE_SEEDS`, never reported). It generates those streams first and then searches `grid` (with `per_dataset_grid` on top) in `configs/detectors.yaml`.
Skip this and the settings in `params` are used.

Tuning is slow in a notebook. Run it in the terminal instead (generation of the tuning streams, then the grid search, commands in the README) and set `TUNE = True` here. The notebook then uses the existing `outputs/tuned_params.yaml`.

In [ ]:
TUNED = f"{OUT}/tuned_params.yaml"
RETUNE = False        # True redoes the tuning even when tuned_params.yaml exists
if TUNE and os.path.exists(TUNED) and not RETUNE:
    print("using the existing", TUNED, "(set RETUNE = True to redo the tuning, or run it in the terminal as described above)")
    print(open(TUNED).read())
elif TUNE:
    run([sys.executable, "scripts/run_composition.py", "--experiment", EXP, "--phase", "tune", "--datasets", *DATASETS] + quick)
    run([sys.executable, "evaluation/run_detection.py", "--phase", "tune", "--experiment", EXP, "--tuned", TUNED], tail=1500)
    print(open(TUNED).read())
elif os.path.exists(TUNED):
    os.remove(TUNED)          # make sure an old tuning result is not used silently
    print("removed an old", TUNED)

## 5. Detect and evaluate
Every detector runs on every stream. The main tolerance is 2% of the time span of the stream, so that logs with very different time scales get a comparable tolerance. A fixed 14 days, and one half, one and two windows of the detector (a window is the PELT window or the mean VC window duration) are the sensitivity analysis. A crashed run counts as a failure (F1 of 0).

In [ ]:
run([sys.executable, "evaluation/run_detection.py", "--phase", "eval", "--experiment", EXP,
     "--tuned", TUNED, "--out-dir", OUT], tail=200)
raw = pd.read_csv(f"{OUT}/detection_raw.csv")
nt = raw["tolerance"].nunique()
print("runs", len(raw) // nt, "| errors", int((raw["status"] == "error").sum() // nt))
if (raw["status"] == "error").any():
    display(raw[raw["status"] == "error"][["dataset", "tier", "seed", "detector", "error"]].drop_duplicates())

## 6. Results (mean and sample standard deviation over seeds)

In [ ]:
s = pd.read_csv(f"{OUT}/detection_summary.csv")
fmt = lambda m, sd: f"{m:.2f} ± {0 if pd.isna(sd) else sd:.2f}"
s["P"]  = [fmt(a, b) for a, b in zip(s.precision_mean, s.precision_std)]
s["R"]  = [fmt(a, b) for a, b in zip(s.recall_mean, s.recall_std)]
s["F1"] = [fmt(a, b) for a, b in zip(s.f1_mean, s.f1_std)]
s["dataset"] = pd.Categorical(s.dataset, DATASETS, ordered=True)
s["tier"] = pd.Categorical(s.tier, TIERS, ordered=True)
for tol in s["tolerance"].unique():
    print("F1 at tolerance", tol)
    display(s[s.tolerance == tol].pivot_table(index=["dataset", "tier"], columns="detector", values="F1",
                                              aggfunc="first", observed=True))

In [ ]:
MAIN = yaml.safe_load(open("configs/detectors.yaml"))["tolerances"][0]["name"]   # the first tolerance in the config
main = s[s.tolerance == MAIN]
display(main.sort_values(["dataset", "tier", "detector"])[["dataset", "tier", "detector", "n_runs", "n_errors", "n_gt", "P", "R", "F1"]])
avg = main.groupby(["tier", "detector"], observed=True).f1_mean.mean().unstack()
print("F1 averaged over the logs (main tolerance)"); display(avg.round(2))
fig, axes = plt.subplots(1, len(DATASETS), figsize=(4.2 * len(DATASETS), 3), sharey=True)
for ax, ds in zip(np.atleast_1d(axes), DATASETS):
    d = main[main.dataset == ds]
    m = d.pivot(index="tier", columns="detector", values="f1_mean").reindex(TIERS)
    e = d.pivot(index="tier", columns="detector", values="f1_std").reindex(TIERS).fillna(0)
    m.plot.bar(yerr=e, ax=ax, ylim=(0, 1.05), capsize=3, legend=ax is np.atleast_1d(axes)[-1])
    ax.set_title(NAMES[ds]); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
np.atleast_1d(axes)[0].set_ylabel("F1"); plt.tight_layout(); plt.show()

### LaTeX table (paper layout, precision, recall and F1 as mean with std in small print, better F1 highlighted with \rg)

In [ ]:
# macro needed in the paper preamble (std with plus-minus sign, no leading zero to save width)
# \newcommand{\resn}[2]{#1{\tiny$\pm$#2}}   and \usepackage{graphicx}, \scriptsize and \setlength{\tabcolsep}{1.5pt} before the table
dets = ["pelt", "vc"]
dnames = {"pelt": "PELT", "vc": "VC"}
def num(x):
    s = "%.2f" % (0.0 if pd.isna(x) else x)
    return s[1:] if s.startswith("0") else s
def res(r, metric):
    return "\\resn{%s}{%s}" % (num(r[metric + "_mean"].iloc[0]), num(r[metric + "_std"].iloc[0]))
def row(ds, tier, d):
    return main[(main.dataset == ds) & (main.tier == tier) & (main.detector == d)]

lines = [r"\resizebox{\linewidth}{!}{%", r"\begin{tabular}{ll" + "ccc" * len(TIERS) + "}", r"\toprule",
         " & & " + " & ".join(r"\multicolumn{3}{c}{\texttt{%s}}" % t for t in TIERS) + r" \\",
         "".join(r"\cmidrule(lr){%d-%d}" % (3 + 3 * i, 5 + 3 * i) for i in range(len(TIERS))),
         "Dataset & Detector & " + " & ".join("P & R & F1" for _ in TIERS) + r" \\", r"\midrule"]
for k, ds in enumerate(DATASETS):
    for j, d in enumerate(dets):
        cells = []
        for tier in TIERS:
            r = row(ds, tier, d)
            if r.empty:
                cells += ["--"] * 3
                continue
            other = row(ds, tier, [x for x in dets if x != d][0])
            best = other.empty or r.f1_mean.iloc[0] >= other.f1_mean.iloc[0] - 1e-9
            f1 = res(r, "f1")
            cells += [res(r, "precision"), res(r, "recall"), r"\rg{%s}" % f1 if best else f1]
        first = r"\multirow{%d}{*}{%s}" % (len(dets), NAMES[ds]) if j == 0 else ""
        lines.append(f"{first} & {dnames[d]} & " + " & ".join(cells) + r" \\")
    lines.append(r"\midrule" if k < len(DATASETS) - 1 else r"\bottomrule")
lines += [r"\end{tabular}}"]
print("\n".join(lines))